In [1]:
import time
start_time = time.perf_counter()
print('Done!')

Done!


In [ ]:
import re
search_dir = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))

In [ ]:
def find_function_source_in_file(path: str, function_name: str) -> str | None:
    sig_re = re.compile(rf'\b{re.escape(function_name)}\b\s*\(')
    def_re = re.compile(
        rf'''^[ \t]*
            (?:[A-Za-z_]\w*(?:\s*\*+)?\s+)+
            \**\s*
            {re.escape(function_name)}\s*\(
        ''',
        re.VERBOSE
    )
    try:
        f = open(path, 'r', errors='ignore')
    except PermissionError:
        return None
    with f:
        collecting = False
        brace_count = 0
        buffer = []
        in_block = False
        for raw in f:
            line = raw
            if in_block:
                end = line.find('*/')
                if end >= 0:
                    in_block = False
                    line = line[end+2:]
                else:
                    continue
            start = line.find('/*')
            if start >= 0:
                end = line.find('*/', start+2)
                if end >= 0:
                    line = line[:start] + line[end+2:]
                else:
                    in_block = True
                    line = line[:start]
            clean = line.split('//',1)[0]
            if clean.rstrip().endswith('\\'):
                continue
            if not collecting:
                if sig_re.search(clean) and def_re.match(clean):
                    sig_raw = [raw]
                    sig_clean = [clean]
                    bal = clean.count('(') - clean.count(')')
                    while bal > 0:
                        nxt_raw = f.readline()
                        if not nxt_raw:
                            break
                        nxt = nxt_raw
                        if in_block:
                            end = nxt.find('*/')
                            if end >= 0:
                                in_block = False
                                nxt = nxt[end+2:]
                            else:
                                continue
                        st = nxt.find('/*')
                        if st >= 0:
                            ed = nxt.find('*/', st+2)
                            if ed >= 0:
                                nxt = nxt[:st] + nxt[ed+2:]
                            else:
                                in_block = True
                                nxt = nxt[:st]
                        nxt_clean = nxt.split('//',1)[0]
                        sig_raw.append(nxt_raw)
                        sig_clean.append(nxt_clean)
                        bal += nxt_clean.count('(') - nxt_clean.count(')')
                    if ''.join(sig_clean).strip().endswith(';'):
                        continue
                    buffer = sig_raw.copy()
                    if '{' in sig_clean[-1]:
                        collecting = True
                        brace_count = sig_clean[-1].count('{') - sig_clean[-1].count('}')
                    else:
                        for body_raw in f:
                            body = body_raw
                            if in_block:
                                end = body.find('*/')
                                if end >= 0:
                                    in_block = False
                                    body = body[end+2:]
                                else:
                                    continue
                            st = body.find('/*')
                            if st >= 0:
                                ed = body.find('*/', st+2)
                                if ed >= 0:
                                    body = body[:st] + body[ed+2:]
                                else:
                                    in_block = True
                                    body = body[:st]
                            body_clean = body.split('//',1)[0]
                            buffer.append(body_raw)
                            if '{' in body_clean:
                                collecting = True
                                brace_count = (
                                    body_clean.count('{') -
                                    body_clean.count('}')
                                )
                                break
                    if not collecting:
                        buffer = []
            else:
                buffer.append(raw)
                tmp = raw
                if in_block:
                    end = tmp.find('*/')
                    if end >= 0:
                        in_block = False
                        tmp = tmp[end+2:]
                    else:
                        continue
                st = tmp.find('/*')
                if st >= 0:
                    ed = tmp.find('*/', st+2)
                    if ed >= 0:
                        tmp = tmp[:st] + tmp[ed+2:]
                    else:
                        in_block = True
                        tmp = tmp[:st]
                tmp_clean = tmp.split('//',1)[0]
                brace_count += tmp_clean.count('{') - tmp_clean.count('}')
                if brace_count == 0:
                    return ''.join(buffer)
    return None

def find_function_native(root_dir: str, function_name: str) -> str | None:
    pat = rf'{re.escape(function_name)}[[:space:]]*\('
    try:
        out = subprocess.check_output([
            'grep', '-RlE',
            '--include=*.c', pat, root_dir
        ], text=True, stderr=subprocess.DEVNULL)
    except subprocess.CalledProcessError:
        return None
    for file_path in out.splitlines():
        src = find_function_source_in_file(file_path, function_name)
        if src:
            return src
    return None

In [2]:
import re
from collections import deque

the_directory = 'output/demo'
ending = 'XXXTHISENDSHEREXXX'
md_file    = 'the_start_markdown.txt'
src_file   = 'the_functions_all.txt'
out_file   = 'in_between_list.txt'

# 1) Parse Markdown tree jadi list pasangan (parent, child)
pairs = []
stack = []
with open(f'{the_directory}/{md_file}') as f:
    for line in f:
        if not line.strip(): continue
        indent = len(line) - len(line.lstrip(' '))
        func   = line.lstrip(' -').strip()
        while stack and stack[-1][0] >= indent:
            stack.pop()
        if stack:
            pairs.append((stack[-1][1], func))
        stack.append((indent, func))

# 2) Load semua blok source dan build adjacency list
with open(f'{the_directory}/{src_file}') as f:
    content = f.read()

pattern = rf"Source Code for\s+([\w_]+)\s*:\s*\n(.*?)(?={re.escape(ending)})"
blocks  = dict(re.findall(pattern, content, flags=re.DOTALL))
adj     = {fn: set() for fn in blocks}

for fn, src in blocks.items():
    for callee in blocks:
        if callee != fn and re.search(rf'\b{re.escape(callee)}\b', src):
            adj[fn].add(callee)

# 3) BFS tiap pasangan
results = []
for parent, child in pairs:
    if parent not in adj or child not in blocks:
        results.append([parent, f"(no source for {parent} or {child})"])
        continue
    # direct call?
    if child in adj[parent]:
        results.append([parent, child])
        continue

    # BFS multi-hop
    visited = {parent}
    queue   = deque([[parent]])
    found   = None
    while queue and not found:
        path = queue.popleft()
        last = path[-1]
        for nb in adj[last]:
            if nb in visited: 
                continue
            visited.add(nb)
            new_path = path + [nb]
            if nb == child:
                found = new_path
                break
            queue.append(new_path)
    if found:
        results.append(found)
    else:
        results.append([parent, f"(no path to {child})"])

# 4) Tulis hasil
with open(f"{the_directory}/{out_file}", 'w') as f:
    for chain in results:
        f.write(', '.join(chain) + '\n')

print('Done!')


Done!


In [3]:
end_time = time.perf_counter()
elapsed = end_time - start_time
hours   = int(elapsed // 3600)
minutes = int((elapsed % 3600) // 60)
seconds = elapsed % 60

print(f"Elapsed time: {hours}h {minutes}m {seconds:.6f}s")
print('Done!')

Elapsed time: 0h 1m 42.325754s
Done!


In [ ]:
import os
import re
import subprocess
from collections import deque

the_directory = 'output/demo'
md_file    = 'the_start_markdown.txt'
src_file   = 'the_functions_all.txt'
out_file   = 'in_between_list.txt'
search_dir = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))

def find_function_source_in_file(path: str, function_name: str) -> str | None:
    sig_re = re.compile(rf'\b{re.escape(function_name)}\b\s*\(')
    def_re = re.compile(
        rf'''^[ \t]*
            (?:[A-Za-z_]\w*(?:\s*\*+)?\s+)+
            \**\s*
            {re.escape(function_name)}\s*\(
        ''',
        re.VERBOSE
    )
    try:
        f = open(path, 'r', errors='ignore')
    except PermissionError:
        return None
    with f:
        collecting = False
        brace_count = 0
        buffer = []
        in_block = False
        for raw in f:
            line = raw
            if in_block:
                end = line.find('*/')
                if end >= 0:
                    in_block = False
                    line = line[end+2:]
                else:
                    continue
            start = line.find('/*')
            if start >= 0:
                end = line.find('*/', start+2)
                if end >= 0:
                    line = line[:start] + line[end+2:]
                else:
                    in_block = True
                    line = line[:start]
            clean = line.split('//',1)[0]
            if clean.rstrip().endswith('\\'):
                continue
            if not collecting:
                if sig_re.search(clean) and def_re.match(clean):
                    sig_raw = [raw]
                    bal = clean.count('(') - clean.count(')')
                    while bal > 0:
                        nxt_raw = f.readline()
                        if not nxt_raw:
                            break
                        sig_raw.append(nxt_raw)
                        nxt = nxt_raw.split('//',1)[0]
                        bal += nxt.count('(') - nxt.count(')')
                    if ''.join([l.split('//',1)[0] for l in sig_raw]).strip().endswith(';'):
                        continue
                    buffer = sig_raw.copy()
                    if '{' in sig_raw[-1]:
                        collecting = True
                        brace_count = sig_raw[-1].count('{') - sig_raw[-1].count('}')
                    else:
                        for body_raw in f:
                            buffer.append(body_raw)
                            body = body_raw.split('//',1)[0]
                            if '{' in body:
                                collecting = True
                                brace_count = body.count('{') - body.count('}')
                                break
                    if not collecting:
                        buffer = []
            else:
                buffer.append(raw)
                tmp = raw
                if in_block:
                    end = tmp.find('*/')
                    if end >= 0:
                        in_block = False
                        tmp = tmp[end+2:]
                    else:
                        continue
                st = tmp.find('/*')
                if st >= 0:
                    ed = tmp.find('*/', st+2)
                    if ed >= 0:
                        tmp = tmp[:st] + tmp[ed+2:]
                    else:
                        in_block = True
                        tmp = tmp[:st]
                tmp_clean = tmp.split('//',1)[0]
                brace_count += tmp_clean.count('{') - tmp_clean.count('}')
                if brace_count == 0:
                    return ''.join(buffer)
    return None

def find_function_native(root_dir: str, function_name: str) -> str | None:
    pat = rf'{re.escape(function_name)}[[:space:]]*\('
    try:
        out = subprocess.check_output([
            'grep', '-RlE', '--include=*.c', pat, root_dir
        ], text=True, stderr=subprocess.DEVNULL)
    except subprocess.CalledProcessError:
        return None
    for file_path in out.splitlines():
        src = find_function_source_in_file(file_path, function_name)
        if src:
            return src
    return None

# 1) Parse markdown jadi list pasangan parent→child
pairs = []
stack = []
with open(f'{the_directory}/{md_file}') as f:
    for line in f:
        if not line.strip(): continue
        indent = len(line) - len(line.lstrip(' '))
        name   = line.lstrip(' -').strip()
        while stack and stack[-1][0] >= indent:
            stack.pop()
        if stack:
            pairs.append((stack[-1][1], name))
        stack.append((indent, name))

# 2) Load blok sumber yang sudah ada
with open(f"{the_directory}/{src_file}") as f:
    content = f.read()
ending = 'XXXTHISENDSHEREXXX'
pattern = rf"Source Code for\s+([\w_]+)\s*:\s*\n(.*?)(?={re.escape(ending)})"
function_sources = dict(re.findall(pattern, content, flags=re.DOTALL))

# 3) Tambah fungsi dari markdown kalau belum ada
all_funcs = {fn for p in pairs for fn in p}
for fn in all_funcs:
    if fn not in function_sources:
        src = find_function_native(search_dir, fn)
        if src:
            function_sources[fn] = src

# 4) Iterasi closure: cari semua callee dalam setiap src dan tambahkan via native search
while True:
    baru = False
    for fn, src in list(function_sources.items()):
        callee_names = set(re.findall(r'\b([A-Za-z_]\w*)\b', src))
        for callee in callee_names:
            if callee not in function_sources:
                src2 = find_function_native(search_dir, callee)
                if src2:
                    function_sources[callee] = src2
                    baru = True
    if not baru:
        break

# 5) Build adjacency list
graph_nodes = set(function_sources)
adj = {fn: set() for fn in graph_nodes}
for fn, src in function_sources.items():
    for callee in graph_nodes:
        if callee != fn and re.search(rf'\b{re.escape(callee)}\b', src):
            adj[fn].add(callee)

# 6) BFS untuk tiap parent→child
results = []
for parent, child in pairs:
    if parent not in adj or child not in graph_nodes:
        results.append([parent, f"(no source for {parent} or {child})"])
        continue
    if child in adj[parent]:
        results.append([parent, child])
        continue
    visited = {parent}
    queue   = deque([[parent]])
    found   = None
    while queue and not found:
        path = queue.popleft()
        for nb in adj[path[-1]]:
            if nb in visited: continue
            visited.add(nb)
            newp = path + [nb]
            if nb == child:
                found = newp
                break
            queue.append(newp)
    results.append(found or [parent, f"(no path to {child})"])

# 7) Tulis output
with open(f"{the_directory}/{out_file}", 'w') as f:
    for chain in results:
        f.write(', '.join(chain) + '\n')

print("Done!")
